<a href="https://colab.research.google.com/github/raffaellaalpaca-ai/walmart-hurricane-project/blob/main/Walmart_Midterm_Notebook_(DS_2002).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Midterm Project

**Group Members: Emma Kee (hvh6ng), Kylie Sharp (dhq4mr), Raffaella Alpaca (bet3at), Miriam Ahmadd (zzx6jb)**

There are three related datasets:
1.   store_locations.csv — store information and geographic details
2.   product_catalog.csv - product information
3.   walmart_transactions.csv — individual sales transactions

These datasets are connected through sku and store_id

#### *Import Data*

In [1]:
# Download a copy of the Git repository to local environment
!git clone https://github.com/ekee7/walmart-hurricane-project.git

fatal: destination path 'walmart-hurricane-project' already exists and is not an empty directory.


In [2]:
import subprocess

REPO_PATH = '/content/walmart-hurricane-project' # Your local clone of the team repo

In [3]:
import pandas as pd

# Import data set from GitHub
url = 'https://raw.githubusercontent.com/ekee7/walmart-hurricane-project/refs/heads/main/data/store_locations.csv'
df_store = pd.read_csv(url)
df_store_original = df_store.copy()

url = 'https://raw.githubusercontent.com/ekee7/walmart-hurricane-project/refs/heads/main/data/product_catalog.csv'
df_product = pd.read_csv(url)
df_product_original = df_product.copy()

url = 'https://raw.githubusercontent.com/ekee7/walmart-hurricane-project/refs/heads/main/data/walmart_transactions.csv'
df_transaction = pd.read_csv(url)
df_transaction_original = df_transaction.copy()

#### *Initial Look at Shape, Data Type,  Missing Values, and Duplicate Rows*

In [4]:
datasets = {
    "Stores": df_store,
    "Products": df_product,
    "Transactions": df_transaction
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


--- Stores ---
Shape: (14, 8)

Data types:
store_id       object
store_name     object
city           object
state          object
zip_code        int64
latitude      float64
longitude     float64
region         object
dtype: object

Missing values:
store_id      0
store_name    0
city          0
state         0
zip_code      0
latitude      1
longitude     1
region        0
dtype: int64

Duplicate rows: 0

--- Products ---
Shape: (45, 7)

Data types:
sku              object
product_name     object
category         object
subcategory      object
unit_cost       float64
supplier         object
package_size     object
dtype: object

Missing values:
sku              0
product_name     0
category         0
subcategory     13
unit_cost        5
supplier         0
package_size     0
dtype: int64

Duplicate rows: 0

--- Transactions ---
Shape: (25562, 8)

Data types:
transaction_id    object
store_id          object
timestamp         object
sku               object
product_name      object
c

#### *Decision Log*

In [5]:
DECISIONS = []

# Function for decision logging
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

### **Data Cleaning for store_locations.csv**

In [6]:
df_store.head()

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


Standardize all text columns to lowercase

In [7]:
text_cols = ["store_id", "store_name", "city", "state", "region"]

for col in text_cols:
    df_store[col] = (
        df_store[col]
        .str.strip()
        .str.lower()
    )

Look at the datatypes of the columns

In [8]:
print('Column datatypes:\n', df_store.dtypes)

Column datatypes:
 store_id       object
store_name     object
city           object
state          object
zip_code        int64
latitude      float64
longitude     float64
region         object
dtype: object


In [9]:
# Change zip_code data type from integer to a string because ZIP codes are dentifiers, not quantities
df_store['zip_code'] = df_store['zip_code'].astype(str)

Standardize the store_id column

In [10]:
df_store["store_id"] = (
    df_store["store_id"]
    .str.replace(r"[^a-z0-9]", "", regex=True)
    .str.replace(r"^([a-z]+)(\d+)$", r"\1-\2", regex=True)
)

In [11]:
df_store['store_id'].unique()

array(['fl-239', 'fl-105', 'fl-330', 'fl-412', 'fl-287', 'fl-156',
       'fl-501', 'fl-078', 'fl-619', 'fl-445', 'fl-302', 'fl-710'],
      dtype=object)

Standardize state: FL

In [12]:
# Standardize state abbreviations
df_store['state'] = df_store['state'].replace({
    'Florida': 'fl',
    'Fla.': 'fl'
})

In [13]:
df_store['state'].unique()

array(['fl', 'florida', 'fla.'], dtype=object)

In [14]:
# Check for duplicate store IDs
duplicate_store_ids = df_store['store_id'].duplicated().sum()

print('Duplicate store IDs:', duplicate_store_ids)

Duplicate store IDs: 2


Drop duplicate rows: We decided to drop duplicate rows because they represent repeated records that convey and store the same information. Because we added a store_id_clean column, we just dropped rows based on store_id because it represents unique store information.

In [15]:
def drop_duplicate_rows(df):
  rows_affected = df.duplicated().sum()
  df.drop_duplicates(inplace=True)

  return df, rows_affected

In [16]:
df_store, rows_affected = drop_duplicate_rows(df_store)

log(
    'Data cleaning',
    'Drop duplicate rows',
    rows_affected
)

[Data cleaning] Drop duplicate rows (2 row(s))


In [17]:
print('Number of rows dropped:', rows_affected)

Number of rows dropped: 2


The number of rows dropped match exactly the number of duplicate store ID values. This validates that the row drop was successful.

### **Data Cleaning for product_catalog.csv**

In [18]:
df_product.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


Standardize all text columns to lowercase

In [19]:
text_cols = [
    "sku",
    "product_name",
    "category",
    "subcategory",
    "supplier",
    "package_size"
]

for col in text_cols:
    df_product[col] = (
        df_product[col]
        .str.strip()
        .str.lower()
    )

Standardize the category column

In [20]:
df_product["category"].value_counts(dropna=False)

,count
category,
emergency,9
entertainment,5
grocery,5
electronics,4
household,4
emergency supplies,3
breakfast,3
baby,3
snacks,2


In [21]:
print('Unique Category Values Before Mapping:\n', df_product['category'].unique())
print('Number of Category Values Before Mapping:', df_product["category"].value_counts(dropna=False).sum())

Unique Category Values Before Mapping:
 ['breakfast foods' 'breakfast' 'emergency supplies' 'emergency'
 'alcoholic beverages' 'alcohol' 'entertainment' 'electronics' 'groceries'
 'grocery' 'health' 'snacks' 'household' 'baby' 'baby products']
Number of Category Values Before Mapping: 45


In [22]:
# Mapping
df_product['category'] = df_product['category'].replace({
    'grocery': 'groceries',
    'breakfast': 'breakfast foods',
    'household': 'household essentials',
    'alcohol': 'alcoholic beverages',
    'emergency': 'emergency supplies',
    'entertainment': 'games & entertainment',
    'baby': 'baby products',
    'health': 'health & wellness'
})

In [23]:
print('Unique Category Values After Mapping:\n', df_product['category'].unique())
print('Number of Category Values After Mapping:', df_product["category"].value_counts(dropna=False).sum())

Unique Category Values After Mapping:
 ['breakfast foods' 'emergency supplies' 'alcoholic beverages'
 'games & entertainment' 'electronics' 'groceries' 'health & wellness'
 'snacks' 'household essentials' 'baby products']
Number of Category Values After Mapping: 45


Fill in the missing subcategory values

In [24]:
df_product['subcategory'].value_counts(dropna=False)

# There are 13 missing values in the subcategory column

,count
subcategory,
NaN,13
aa,3
pop-tarts,2
bottled,2
streaming,2
playing,2
peanut,2
bread,2
candles,2


In [25]:
df_product['subcategory'].unique()

array([nan, 'pop-tarts', 'bottled', 'aa', 'flashlight', 'light',
       'playing', 'streaming', 'canned', 'bread', 'peanut', 'candles',
       'first', 'portable', 'chips', 'paper', 'toilet', 'diapers', 'baby',
       'tarp'], dtype=object)

In [26]:
# Investigate the subcategory column's missing values
df_product[df_product['subcategory'].isna()][
    ['sku', 'product_name', 'category', 'subcategory']
]

,sku,product_name,category,subcategory
0,pt-12,pop-tarts strawberry,breakfast foods,NaN
2,poptart-straw,pop-tarts strawberry,breakfast foods,NaN
6,wtr_24pk,water - 24 pack,emergency supplies,NaN
11,led-flash,flashlight led,emergency supplies,NaN
12,beer-24,light beer (24-pack),alcoholic beverages,NaN
14,bg-mono,monopoly board game,games & entertainment,NaN
15,monopoly-01,monopoly board game,games & entertainment,NaN
16,game-mono,board game - monopoly,games & entertainment,NaN
22,cnsoup-01,canned soup,groceries,NaN
30,firstaid-kit,first aid kit,health & wellness,NaN


In [27]:
# Investigate the subcategory column's shortened subcategory names
df_product[['product_name', 'category', 'subcategory']].sort_values('subcategory')

,product_name,category,subcategory
7,batteries aa 4pk,emergency supplies,aa
8,aa batteries 4-pack,emergency supplies,aa
9,aa batteries 4-pack,emergency supplies,aa
41,infant formula,baby products,baby
4,bottled water 24pk,emergency supplies,bottled
5,bottled water 24-pack,emergency supplies,bottled
24,white bread,groceries,bread
23,white bread,groceries,bread
28,emergency candles,emergency supplies,candles
27,candles (emergency),emergency supplies,candles


The subcategory column contained 13 missing values. After examining the corresponding product names and categories, we determined that these missing values could generally be inferred from other records with matching or similar product descriptions.

In [28]:
# Assign subcategories to fill the missing values
df_product.loc[0, "subcategory"] = "pop-tarts"
df_product.loc[2, "subcategory"] = "pop-tarts"
df_product.loc[6, "subcategory"] = "bottled"
df_product.loc[11, "subcategory"] = "flashlight"
df_product.loc[12, "subcategory"] = "light"
df_product.loc[[14, 15, 16], "subcategory"] = "board games" # Created a new Board Games subcategory for Monopoly products
df_product.loc[22, "subcategory"] = "canned"
df_product.loc[30, "subcategory"] = "first"
df_product.loc[31, "subcategory"] = "portable"
df_product.loc[42, "subcategory"] = "baby"
df_product.loc[43, "subcategory"] = "tarp"

In [29]:
# Confirmation that there are no more NaN values
df_product["subcategory"].isna().sum()

np.int64(0)

Standardize the product_name column

In [30]:
df_product["product_name"].value_counts(dropna=False)

,count
product_name,
pop-tarts strawberry,3
monopoly board game,2
aa batteries 4-pack,2
white bread,2
peanut butter,2
light beer (24-pack),2
flashlight led,2
playing cards,2
paper towels,2


In [31]:
# Create a mapping
df_product["product_name"] = df_product["product_name"].replace({
    "pop-tarts strawberry": "strawberry pop-tart",
    "water - 24 pack": "bottled water 24-pack",
    "batteries aa 4pk": "aa batteries 4-pack",
    "board game - monopoly": "monopoly board game",
    "first-aid kit": "first aid kit",
    "tarp - heavy duty": "heavy duty tarp",
    "diapers sz 3": "baby diapers - size 3",
    "bottled water 24pk": "bottled water 24-pack",
    "soup - canned": "canned soup",
    "candles (emergency)": "emergency candles",
    "variety pack - chips": "chips variety pack",
    "toilet paper 12pk": "toilet paper 12-pack"
})

In [32]:
df_product["product_name"].value_counts(dropna=False)

,count
product_name,
strawberry pop-tart,4
bottled water 24-pack,3
aa batteries 4-pack,3
monopoly board game,3
flashlight led,2
light beer (24-pack),2
playing cards,2
canned soup,2
baby diapers - size 3,2


In [33]:
# Investigate similar product names
df_product[
    df_product["product_name"].isin([
        "portable phone charger",
        "portable charger",
        "streaming device",
        "roku streaming stick",
        "infant formula",
        "baby formula",
        "toilet paper 12-roll",
        "toilet paper 12-pack"
    ])
].sort_values("product_name")

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
42,babyformula,baby formula,baby products,baby,18.25,sara lee,6 pack
41,bform-01,infant formula,baby products,baby,18.87,procter & gamble,4-pack
31,chrg-port,portable charger,electronics,portable,11.99,georgia-pacific,single
32,portcharger,portable phone charger,electronics,portable,15.03,johnson & johnson,12 pack
19,stream-01,roku streaming stick,electronics,streaming,16.70,3m company,12 pack
20,roku-std,streaming device,electronics,streaming,16.99,anheuser-busch,12 pack
37,tp-12,toilet paper 12-pack,household essentials,toilet,5.58,campbell's,1 pack
38,toiletpaper12,toilet paper 12-roll,household essentials,toilet,5.75,j.m. smucker,24-pack


Note: We kept portable phone charger and portable charger, streaming device and roku streaming stick, infant formula and baby formula, and toilet paper 12-roll and toilet paper 12-pack as separate products.

Investigate missing values across df_product

In [34]:
df_product.isna().sum()

,0
sku,0
product_name,0
category,0
subcategory,0
unit_cost,5
supplier,0
package_size,0


Note: There are 5 missing unit_cost values

Standardize the package_size column's formatting

In [35]:
df_product["package_size"] = (
    df_product["package_size"]
    .str.replace("-", " ", regex=False)
    .str.replace("_", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*(pk|pack)$", "-pack", regex=True)
)

In [36]:
df_product["package_size"].value_counts(dropna=False)

,count
package_size,
4-pack,10
24-pack,9
1-pack,7
single,6
6-pack,5
12-pack,5
bundle,3


Note: Not all of the package_size values are in the format of _-pack. There are single and bundle values.

Check for duplicate SKU values

In [37]:
df_product[df_product["sku"].duplicated(keep=False)].sort_values("sku")

# There are no duplicate SKU values

,sku,product_name,category,subcategory,unit_cost,supplier,package_size


### **Data Cleaning for walmart_transactions.csv**

In [38]:
df_transaction.head()

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-24T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,fl-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82


Standardize text columns to lowercase

In [39]:
text_cols = [
    "transaction_id",
    "store_id",
    "sku",
    "product_name",
    "category"
]

for col in text_cols:
    df_transaction[col] = (df_transaction[col].str.strip().str.lower())

Look at the datatypes of the columns

In [40]:
print('Column datatypes:\n', df_transaction.dtypes)

Column datatypes:
 transaction_id    object
store_id          object
timestamp         object
sku               object
product_name      object
category          object
quantity           int64
unit_price        object
dtype: object


Update unit_price to be float type, remove $, and strip leading/trailing whitespace

In [41]:
# unit_price is a String but it should be a float type for future calculations (i.e. summing/etc)

standardized = df_transaction['unit_price'].str.strip().str.replace('$', '', regex=False).astype(float)
rows_affected = df_transaction['unit_price'].ne(standardized).sum()
df_transaction['unit_price'] = standardized

log(
    'Data cleaning',
    'unit_price: update to be float type and remove $',
    rows_affected
)

[Data cleaning] unit_price: update to be float type and remove $ (25562 row(s))


In [42]:
# Post to_float analysis

print('unit_price Data Type:', df_transaction['unit_price'].dtype)

unit_price Data Type: float64


Transform timestamp column data type to datetime object

In [43]:
# Intitial analysis

print('Pre-parse_timestamp()\n')
print('Unique values', df_transaction['timestamp'].unique())
print('Has NaN values:', df_transaction['timestamp'].hasnans)
print('NaN count:', df_transaction['timestamp'].isna().sum())

Pre-parse_timestamp()

Unique values ['2024-09-24T09:48:13' '2024-09-13 20:30:20' '2024-09-17 10:06:00' ...
 '09-26-2024 08:47:08' '16-Sep-2024 16:39' '2024-09-25 17:08:35']
Has NaN values: True
NaN count: 886


In [44]:
# Parse timestamp
standardized = pd.to_datetime(df_transaction["timestamp"], errors='coerce', format='mixed')
rows_affected = (df_transaction["timestamp"].notna() & standardized.isna()).sum()
df_transaction["timestamp"] = standardized

log(
    'Data cleaning',
    'timestamp: Parse timestamp by converting to date time format',
    rows_affected
)

[Data cleaning] timestamp: Parse timestamp by converting to date time format (0 row(s))


In [45]:
# Confirmation

print('Post-parse_timestamp()\n')
print('Updated datatype:', df_transaction['timestamp'].dtype)
print('NaT count:', df_transaction['timestamp'].isna().sum()) # NaN -> NaT count did not change
print('Unique values:\n', df_transaction['timestamp'].unique())

Post-parse_timestamp()

Updated datatype: datetime64[ns]
NaT count: 886
Unique values:
 <DatetimeArray>
['2024-09-24 09:48:13', '2024-09-13 20:30:20', '2024-09-17 10:06:00',
 '2024-09-11 06:29:08', '2024-09-26 15:59:42', '2024-09-23 10:32:34',
 '2024-09-26 09:26:00', '2024-09-22 14:47:00', '2024-09-08 16:40:14',
 '2024-09-24 20:03:00',
 ...
 '2024-09-24 17:10:49', '2024-09-19 13:59:16', '2024-09-07 22:22:26',
 '2024-09-11 20:40:00', '2024-09-05 13:13:40', '2024-09-04 17:32:37',
 '2024-09-03 18:28:07', '2024-09-26 08:47:08', '2024-09-16 16:39:00',
 '2024-09-25 17:08:35']
Length: 23418, dtype: datetime64[ns]


Originally, we had a standardized = pd.to_datetime(df[column_name], errors='coerce'). errors='coerce' tells Pandas to convert valid timestamp strings into datetime objects, convert missing values (NaN, None, etc) into NaT, and convert unparseable timestamp values into NaT. This 3rd case is what made our NA value count go from 884 to 23028 post-function application, meaning that there were many cases where the original timestamp values were not NaN but were converted to NaT. In other words, many of the original timestamp values may not be compatible with Pandas' datetime parsing rules.

To better handle these cases, we decided to use a format='mixed' parameter. This lets Pandas infer the format of each value individually, which is useful because our dataset contains multiple timestamp formats.

Furthermore, to maintain the original timestamp values for comparison, we created a copy of the column using .copy() before parsing. This allowed us to distinguish between values that were already missing and values that failed to parse during conversion. We then updated the rows_affected calculation to count only values that were originally non-missing but became NaT after parsing, giving us a more accurate measure of parsing failures.

Check for missing values

In [46]:
df_transaction.isna().sum()

,0
transaction_id,0
store_id,0
timestamp,886
sku,0
product_name,0
category,0
quantity,0
unit_price,0


Look for transaction_id duplicates

In [47]:
df_transaction["transaction_id"].duplicated().sum()

np.int64(30)

In [48]:
df_transaction[
    df_transaction["transaction_id"].duplicated(keep=False)
].sort_values("transaction_id")

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
19474,txn000961,fl-078,2024-09-13 19:16:56,bread-wh,white bread,grocery,2,2.55
5326,txn000961,fl-078,2024-09-13 19:16:56,bread-wh,white bread,grocery,2,2.55
17990,txn003638,fl-501,2024-09-17 11:12:11,beer-24,beer light 24pk,alcohol,2,19.63
3131,txn003638,fl-501,2024-09-17 11:12:11,beer-24,beer light 24pk,alcohol,2,19.63
21161,txn004028,fl-239,2024-09-18 22:52:24,wtr-24,bottled water 24pk,emergency,2,5.17
541,txn004028,fl-239,2024-09-18 22:52:24,wtr-24,bottled water 24pk,emergency,2,5.17
15831,txn004232,fl-330,2024-09-18 08:49:38,bform-01,infant formula,baby,2,30.08
2937,txn004232,fl-330,2024-09-18 08:49:38,bform-01,infant formula,baby,2,30.08
19553,txn004258,fl-412,2024-09-18 16:49:00,bg-mono,monopoly board game,entertainment,2,20.34
9972,txn004258,fl-412,2024-09-18 16:49:00,bg-mono,monopoly board game,entertainment,2,20.34


Drop duplicate rows: We decided to drop duplicate rows because they represent repeated records that could inflate transaction counts and sales totals. Only rows that are identical across all columns are removed.

In [49]:
has_duplicates = df_transaction['transaction_id'].duplicated().any()
is_unique = df_transaction['transaction_id'].is_unique

print('transaction_id has duplicates?:', has_duplicates)
print('transaction_id has all unique?:', is_unique)

# transaction_id has no duplicate values and has all unique values
# This column does not need to be cleaned

transaction_id has duplicates?: True
transaction_id has all unique?: False


In [50]:
df_transaction, rows_affected = drop_duplicate_rows(df_transaction)

log(
    'Data cleaning',
    'Drop duplicate rows',
    rows_affected
)

[Data cleaning] Drop duplicate rows (30 row(s))


In [51]:
df_transaction["transaction_id"].duplicated().sum()

np.int64(0)

Standardize formatting of store_id in df_transaction to match store_id in df_store

In [52]:
# df_store's store_id uses hyphens
# df_transaction's store_id uses underscores

df_transaction["store_id"] = (
    df_transaction["store_id"]
    .str.replace(r"[^a-z0-9]", "", regex=True)
    .str.replace(r"^([a-z]+)(\d+)$", r"\1-\2", regex=True)
)

Changing negative quantity values to positive quantity values: We decided to make the quantity values positive because negative quantities do not make sense in this context. Instead of dropping the rows, we decided to view negative quantities as user entry mistakes. We converted negative quantity values to positive values using abs() so all quantities represent nonnegative units sold.

In [53]:
def neg_to_pos(df, column_name):
  rows_affected = (df[column_name] < 0).sum()
  df[column_name] = df[column_name].abs()

  return df, rows_affected

In [54]:
df_transaction, rows_affected = neg_to_pos(df_transaction, 'quantity')

log(
    'Data cleaning',
    'quantity: Changed negative quantity values to positive values',
    rows_affected
)

[Data cleaning] quantity: Changed negative quantity values to positive values (130 row(s))


Check for invalid prices in unit_price

In [55]:
invalid_prices = df_transaction[df_transaction['unit_price'].isna() | (df_transaction['unit_price'] <= 0)]

print('Number of Invalid Prices:', len(invalid_prices))
print(invalid_prices[['transaction_id', 'sku', 'unit_price']])

Number of Invalid Prices: 0
Empty DataFrame
Columns: [transaction_id, sku, unit_price]
Index: []


### **Cross Examine All Three Datasets**

In [56]:
print("SKUs not in df_product:",
      (~df_transaction["sku"].isin(df_product["sku"])).sum())

print("Store IDs not in df_store:",
      (~df_transaction["store_id"].isin(df_store["store_id"])).sum())

SKUs not in df_product: 0
Store IDs not in df_store: 0


We are going to use df_product as the source of truth for product names. Because each df_transaction SKU matches a SKU in df_product, we are going to use df_product to standardize product_name and category in df_transaction. This ensures that the same SKU has the same product name and category in both the df_product and df_transaction datasets.

In [57]:
df_transaction["product_name"] = df_transaction["sku"].map(
    df_product.set_index("sku")["product_name"]
)

df_transaction["category"] = df_transaction["sku"].map(
    df_product.set_index("sku")["category"]
)

Missing value notes:
1.   df_store: 1 missing latitude and 1 missing longitude
2.   df_product: 5 missing unit costs
3.   df_transaction: 884 missing timestamps

For now, we kept all of these values as NaN. In the future, if we do need the latitude/longitude of that one store, we can look up the information online. Additionally, during analysis that involves dates or times, we should exclude the 884 rows that are missing timestamps.

### **Create one combined DataFrame by joining df_transactions with df_product and df_store**

We used left join to combine our Walmart transactions dataset with the product catalog and store locations datasets. This allows us to keep all 25,532 transaction records while adding relevant product information (such as product name, category, supplier, and package size) and store information (such as city, state, and region). Even if a transaction does not have a matching product or store record, the transaction itself is preserved, and the missing information is represented as NaN.

In [58]:
# Adds product information using sku
df = df_transaction.merge(
    df_product,
    on="sku",
    how="left",
    suffixes=("", "_product")
)

# Adds store information using store_id
df = df.merge(
    df_store,
    on="store_id",
    how="left",
    suffixes=("", "_store")
)

In [59]:
# Ensure that df_product["sku"] and df_store["store_id"] are unique
# Both should return true
print('df_product["sku"] Unique?:', df_product["sku"].is_unique)
print('df_store["store_id"] Unique?:', df_store["store_id"].is_unique)

df_product["sku"] Unique?: True
df_store["store_id"] Unique?: True


### **Weather API Join**